# 7. Modelado Predictivo (Modular) — Screening Preanestésico**Objetivo**: Predecir si un paciente requiere valoración preanestésica (`target=1`).**Estrategia Fase 2**: Maximizar ROC-AUC (Ranking) para luego ajustar el umbral operativo.**Población**: Solo pacientes **adultos** (Edad >= 18 años).> [!NOTE]> Este notebook utiliza Smart Imputation, HistGradientBoosting y optimización por ROC-AUC.

## 1. Setup y Carga de Módulos

In [ ]:
import sysimport osimport pandas as pdimport numpy as npimport warnings# Configuraciónwarnings.filterwarnings('ignore')pd.set_option('display.max_columns', None)RANDOM_STATE = 42np.random.seed(RANDOM_STATE)# Agregar directorio raíz al pathcurrent_dir = os.getcwd()if current_dir.endswith('notebook') or current_dir.endswith('notebooks'):    sys.path.append(os.path.abspath('..'))else:    sys.path.append(os.path.abspath('.'))print(f"PYTHONPATH: {sys.path[-1]}")from utils.data_loader import load_opera_completo, load_features_metadatafrom utils.feature_engineering import resolve_feature_columns, get_imputation_strategies, sanitize_features_for_subset, ENCODING_FIX_MAPimport utils.modeling as modimport utils.visualization as vizprint("Modules imported successfully.")

## 2. Carga y Preparación de Datos

In [ ]:
# 2.1 Cargar Dataset Maestrodf = load_opera_completo()print(f"Dataset completo: {df.shape[0]:,} registros")# 2.2 FILTRO: Solo pacientes adultos (Edad >= 18)n_antes = len(df)df = df[df['Edad'] >= 18].reset_index(drop=True)n_despues = len(df)print(f"Filtro adultos (Edad >= 18): {n_antes:,} -> {n_despues:,} registros")print(f"  Pacientes pediátricos excluidos: {n_antes - n_despues:,}")

In [ ]:
# 2.3 Cargar Metadatos de Featuresfeatures_meta = load_features_metadata()selected_names = features_meta['Variable'].tolist()# 2.4 Resolver Columnas de Featuresfeature_columns, missing = resolve_feature_columns(selected_names, df.columns, features_meta)print(f"\nResumen de Features:")print(f"  - Seleccionadas (originales): {len(selected_names)}")print(f"  - Resueltas en DataFrame:     {len(feature_columns)}")if missing:    print(f"Features NO encontradas ({len(missing)}): {missing}")else:    print("Todas las features fueron resueltas correctamente.")

In [ ]:
# 2.5 Corregir Encoding en Nombres de Columnasdf = df.rename(columns=ENCODING_FIX_MAP)feature_columns = [ENCODING_FIX_MAP.get(c, c) for c in feature_columns]# 2.6 Sanitizar Features (Eliminar Constantes en Subset Adultos)print("\n--- Sanitización de Features (Adultos) ---")feature_columns, dropped = sanitize_features_for_subset(df, feature_columns)X = df[feature_columns].copy()y = df['target'].copy()print(f"X shape: {X.shape}")print(f"y shape: {y.shape}")print(f"Prevalencia target=1: {y.mean():.4f} ({y.sum():,} de {len(y):,})")

## 3. Split y Smart Imputation

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplitfrom sklearn.impute import SimpleImputerfrom sklearn.preprocessing import StandardScaler, RobustScalerfrom sklearn.compose import ColumnTransformerfrom sklearn.pipeline import Pipeline# 3.1 Split Estratificadosplitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)train_idx, test_idx = next(splitter.split(X, y))X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")pos_weight_ratio = (y_train == 0).sum() / (y_train == 1).sum()print(f"Ratio de desbalance (scale_pos_weight): {pos_weight_ratio:.4f}")# 3.2 Definir Estrategias de Imputación (Smart Imputation)strategies = get_imputation_strategies(feature_columns)cols_zero = strategies['fill_zero']cols_median = strategies['fill_median']print(f"\nEstrategias de Imputación:")print(f"  - Fill Zero (Scores/Counts): {len(cols_zero)} variables")print(f"  - Fill Median (Medidas):     {len(cols_median)} variables")# 3.3 Construir ColumnTransformerpreprocessor_tree = ColumnTransformer(    transformers=[        ('zero', SimpleImputer(strategy='constant', fill_value=0), cols_zero),        ('median', SimpleImputer(strategy='median'), cols_median)    ],    verbose_feature_names_out=False).set_output(transform="pandas")preprocessor_linear = ColumnTransformer(    transformers=[        ('zero', Pipeline([            ('imputer', SimpleImputer(strategy='constant', fill_value=0)),            ('scaler', StandardScaler())        ]), cols_zero),        ('median', Pipeline([            ('imputer', SimpleImputer(strategy='median')),            ('scaler', RobustScaler())        ]), cols_median)    ],    verbose_feature_names_out=False).set_output(transform="pandas")# Aplicar transformacionesprint("Aplicando transformaciones...")try:    X_train_tree = preprocessor_tree.fit_transform(X_train)    X_test_tree = preprocessor_tree.transform(X_test)    X_train_linear = preprocessor_linear.fit_transform(X_train)    X_test_linear = preprocessor_linear.transform(X_test)    print("Smart Imputation completada.")except Exception as e:    print(f"Error en imputación: {e}")    print("Aplicando fallback a imputación simple...")    imputer = SimpleImputer(strategy='median')    X_train_tree = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)    X_test_tree = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)    X_train_linear = X_train_tree    X_test_linear = X_test_tree

## 4. Entrenamiento y Evaluación Comparativa

In [ ]:
# 4.1 Definir Modelos Basemodels_dict = mod.get_models_definitions(random_state=RANDOM_STATE, scale_pos_weight=pos_weight_ratio)# 4.2 Entrenar y Evaluarresults = {}predictions = {}for name, model in models_dict.items():    print(f"Entrenando {name}...")        if name == 'Regresión Logística':        X_tr, X_te = X_train_linear, X_test_linear    else:        X_tr, X_te = X_train_tree, X_test_tree            y_pred, y_proba, metrics = mod.evaluate_model_performance(        name, model, X_tr, X_te, y_train, y_test    )        results[name] = metrics    predictions[name] = {'y_pred': y_pred, 'y_proba': y_proba}# 4.3 Tabla Comparativadf_results = pd.DataFrame(results).T.sort_values('ROC-AUC', ascending=False)print(f"\n{'='*60}")print("RANKING DE MODELOS (por ROC-AUC) - Solo Adultos")print(f"{'='*60}")print(df_results)

## 5. Visualización de Resultados

In [ ]:
# 5.1 Matrices de Confusiónviz.plot_confusion_matrices(predictions, y_test, save_path='../matrices_confusion_phase2.png')

In [ ]:
# 5.2 Curvas ROC y PRviz.plot_roc_pr_curves(predictions, y_test, save_path='../curvas_roc_pr_phase2.png')

## 6. Optimización (Optuna) del Mejor Modelo

In [ ]:
# 6.1 Identificar el Mejor Modelo (por ROC-AUC)best_model_name = df_results.index[0]best_auc_initial = df_results.iloc[0]['ROC-AUC']print(f"Mejor modelo discriminativo: {best_model_name} (AUC={best_auc_initial:.4f})")print("Iniciando optimización de hiperparámetros (Target: Maximizar ROC-AUC)...")# Seleccionar datosif best_model_name == 'Regresión Logística':    X_train_opt = X_train_linear    X_test_opt = X_test_linearelse:    X_train_opt = X_train_tree    X_test_opt = X_test_tree# 6.2 Optimizar (Target ROC-AUC)study = mod.optimize_best_model(    best_model_name,     X_train_opt, y_train,     random_state=RANDOM_STATE,     scale_pos_weight=pos_weight_ratio,    n_trials=30,     metric='roc_auc')print(f"Mejor AUC (CV): {study.best_value:.4f}")best_params = study.best_paramsprint(f"Mejores parámetros: {best_params}")

In [ ]:
# 6.3 Re-entrenar con mejores parámetrosfrom sklearn.linear_model import LogisticRegressionfrom sklearn.tree import DecisionTreeClassifierfrom sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifierimport xgboost as xgbfinal_model = Noneif best_model_name == 'XGBoost':    final_model = xgb.XGBClassifier(        **best_params,        scale_pos_weight=pos_weight_ratio,        eval_metric='aucpr',        use_label_encoder=False,        random_state=RANDOM_STATE,        n_jobs=-1    )elif best_model_name == 'HistGradientBoosting':    final_model = HistGradientBoostingClassifier(        **best_params,        class_weight='balanced',        scoring='roc_auc',        random_state=RANDOM_STATE    )elif best_model_name == 'Random Forest':    final_model = RandomForestClassifier(        **best_params,        class_weight='balanced',        random_state=RANDOM_STATE,        n_jobs=-1    )elif best_model_name == 'Regresión Logística':    final_model = LogisticRegression(        **best_params,        class_weight='balanced',        solver='lbfgs',        max_iter=1000,        random_state=RANDOM_STATE    )elif best_model_name == 'Árbol de Decisión':    final_model = DecisionTreeClassifier(        **best_params,        class_weight='balanced',        random_state=RANDOM_STATE    )# Evaluación Finaly_pred_opt, y_proba_opt, metrics_opt = mod.evaluate_model_performance(    f'{best_model_name} Optimizado', final_model,     X_train_opt, X_test_opt, y_train, y_test)print(f"\nMejora en AUC: {metrics_opt['ROC-AUC'] - best_auc_initial:.4f}")

In [ ]:
# 6.4 Optimización de Umbral (Post-hoc)opt_thresh, opt_f2, thresholds, f2_scores = mod.find_optimal_threshold(y_test, y_proba_opt)viz.plot_threshold_optimization(thresholds, f2_scores, opt_thresh, save_path='../umbral_optimo_phase2.png')print(f"Umbral Optimo para F2: {opt_thresh:.2f}")print(f"F2-Score Final: {opt_f2:.4f}")

## 7. Interpretabilidad (SHAP)

In [ ]:
import shapimport numpy as npprint(f"Explicando modelo: {type(final_model).__name__}")try:    if best_model_name in ['XGBoost', 'Random Forest', 'Árbol de Decisión', 'HistGradientBoosting']:        explainer = shap.TreeExplainer(final_model)        shap_explanation = explainer(X_test_opt)     elif best_model_name == 'Regresión Logística':        masker = shap.maskers.Independent(data=X_train_opt)        explainer = shap.LinearExplainer(final_model, masker=masker)        shap_explanation = explainer(X_test_opt)    else:        explainer = shap.Explainer(final_model, X_train_opt)        shap_explanation = explainer(X_test_opt)except Exception as e:    print(f"Error inicializando SHAP ({e}). Usando fallback KernelExplainer...")    background = shap.kmeans(X_train_opt, 10)    explainer = shap.KernelExplainer(final_model.predict_proba, background)    shap_explanation = explainer.shap_values(X_test_opt.iloc[:50])    if isinstance(shap_explanation, list): shap_explanation = shap_explanation[1]# Procesar shap_explanationif isinstance(shap_explanation, np.ndarray):    vals = shap_explanationelif hasattr(shap_explanation, 'values'):    vals = shap_explanation.valueselse:    vals = shap_explanationif len(vals.shape) == 3:    vals = vals[:, :, 1]    print(f"SHAP values shape final: {vals.shape}")# 7.1 Global Summaryviz.plot_shap_summary(vals, X_test_opt, save_path='../shap_summary_phase2.png')

In [ ]:
# 7.2 Top Features Dependenceif len(vals) > 0:    shap_importance = np.abs(vals).mean(axis=0)    top_indices = np.argsort(shap_importance)[-5:][::-1]    top_features = [X_test_opt.columns[i] for i in top_indices]    viz.plot_shap_dependence(vals, X_test_opt, top_features, save_path='../shap_dependence_phase2.png')

In [ ]:
# 7.3 Ejemplo Local (Waterfall)tp_mask = (y_test.values == 1) & ((y_proba_opt >= opt_thresh).astype(int) == 1)tp_indices = np.where(tp_mask)[0]if len(tp_indices) > 0:    idx = tp_indices[0]        if hasattr(explainer, 'expected_value'):        ev = explainer.expected_value    else:        ev = None        if isinstance(ev, (list, np.ndarray)) and len(ev) > 1:        ev = ev[1]        viz.plot_shap_waterfall(vals, explainer, X_test_opt, idx, save_path='../shap_waterfall_phase2.png', expected_value=ev)